# Rock-Paper-Scissors: crop + augmentation training (Colab)
split (by class) -> hand crop 256 -> augment train only -> resize 128 -> normalise -> model.
Set the GPU first (Runtime -> Change runtime type -> T4 GPU). Edit `PROJECT` in the settings cell so it points to the folder in the Google Drive you sign in with; that folder must contain the `Paper`, `Rock` and `Scissors` sub-folders.

In [ ]:
!pip install -q wandb

## Libraries and settings

In [ ]:
import os, json, math, random, shutil, time
import numpy as np
import pandas as pd
import cv2
import h5py
import wandb
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (classification_report, f1_score, confusion_matrix, accuracy_score,
                             balanced_accuracy_score, precision_recall_fscore_support)

In [ ]:
PROJECT = "/content/drive/MyDrive/Dataset"
DATA    = "/content/Dataset"
OUT     = "/content"

SEED       = 42
WORK_SIZE  = 256
INPUT_SIZE = 128
MARGIN     = 0.25
MAX_EPOCHS = 80
PATIENCE   = 12
BATCH_SIZE = 32
LR         = 1e-3

WANDB_MODE    = "online"
WANDB_PROJECT = "rps-acv-assignment3"

WORK_FILE  = f"work_{WORK_SIZE}_m{int(MARGIN * 100)}.npz"
SPLIT_FILE = f"{OUT}/split_by_class.csv"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ.setdefault("WANDB_SILENT", "true")
os.makedirs(f"{OUT}/checkpoints", exist_ok=True)

## Google Drive, GPU, W&B

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("dataset folder found:", os.path.isdir(PROJECT), "|", PROJECT)
if not os.path.isdir(PROJECT):
    print("MyDrive contains:", os.listdir('/content/drive/MyDrive'))

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: switch the runtime to T4)")

In [ ]:
import getpass
wandb.login(key=getpass.getpass("W&B API key: "))

## Data and split

In [ ]:
if not os.path.exists(DATA):
    shutil.copytree(PROJECT, DATA, ignore=shutil.ignore_patterns(
        "wandb", "checkpoints", "checkpoints_aug", "*.py", "*.ipynb", "*.npz", "*.csv", "*.pth", "*.h5", "*.json", "crop_preview*"))
CLASSES = [d for d in sorted(os.listdir(DATA)) if os.path.isdir(f"{DATA}/{d}")]
print("classes:", CLASSES)
for name in ["split_by_class.csv", WORK_FILE]:
    if os.path.exists(f"{PROJECT}/{name}") and not os.path.exists(f"{OUT}/{name}"):
        shutil.copy(f"{PROJECT}/{name}", f"{OUT}/{name}")
        print("restored", name)

rows = []
for cls in CLASSES:
    for fname in sorted(os.listdir(f"{DATA}/{cls}")):
        if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
            rows.append({"cls": cls, "file": fname, "rel": f"{cls}/{fname}", "path": f"{DATA}/{cls}/{fname}",
                         "phone": "pixel" if fname.upper().startswith("PXL") else "redmi"})
df = pd.DataFrame(rows)
print(len(df), "images")
print(df["cls"].value_counts().to_dict())

In [ ]:
missing = len(df)
if os.path.exists(SPLIT_FILE):
    df = df.merge(pd.read_csv(SPLIT_FILE)[["rel", "split"]], on="rel", how="left")
    missing = int(df["split"].isna().sum())
    print("split_by_class.csv found, images without an entry:", missing)
if missing > 0:
    df = df.drop(columns=["split"], errors="ignore")
    labels = df["cls"].values
    tr, tmp = train_test_split(np.arange(len(df)), test_size=0.30, stratify=labels, random_state=SEED)
    va, te = train_test_split(tmp, test_size=0.50, stratify=labels[tmp], random_state=SEED)
    df["split"] = "train"
    df.loc[va, "split"] = "val"
    df.loc[te, "split"] = "test"
    df[["rel", "split"]].to_csv(SPLIT_FILE, index=False)
    print("new split made")
print(pd.crosstab(df["split"], df["cls"], margins=True))

## Hand crop

In [ ]:
def load_upright(path, max_side=1024):
    with Image.open(path) as im:
        im.draft("RGB", (max_side, max_side))
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.thumbnail((max_side, max_side), Image.BILINEAR)
        return np.asarray(im)

def hand_crop(rgb, margin=0.25):
    H, W = rgb.shape[:2]
    scale = 320 / max(H, W)
    small = cv2.resize(rgb, (max(1, int(W * scale)), max(1, int(H * scale))), interpolation=cv2.INTER_AREA)
    sh, sw = small.shape[:2]
    ycrcb = cv2.cvtColor(small, cv2.COLOR_RGB2YCrCb)
    mask = cv2.inRange(ycrcb, (0, 133, 77), (255, 173, 127))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    box = None
    if contours:
        areas = [cv2.contourArea(c) for c in contours]
        biggest = max(areas)
        if 0.02 <= biggest / (sh * sw) <= 0.85:
            keep = [c for c, a in zip(contours, areas) if a >= 0.2 * biggest]
            x, y, w, h = cv2.boundingRect(np.concatenate(keep))
            box = (x / scale, y / scale, w / scale, h / scale)
    if box is None:
        side, cx, cy, found = float(min(H, W)), W / 2, H / 2, False
    else:
        x, y, w, h = box
        cx, cy = x + w / 2, y + h / 2
        side, found = min(max(w, h) * (1 + 2 * margin), float(min(H, W))), True
    x0 = int(round(min(max(cx - side / 2, 0), W - side)))
    y0 = int(round(min(max(cy - side / 2, 0), H - side)))
    side = int(round(side))
    return rgb[y0:y0 + side, x0:x0 + side], found, (x0, y0, side)

def make_work_copy(rgb, margin=MARGIN):
    crop, found, _ = hand_crop(rgb, margin)
    return np.array(Image.fromarray(crop).resize((WORK_SIZE, WORK_SIZE), Image.LANCZOS)), found

def preview_crops(table, n_per_class=6, seed=0, per_row=3):
    picks = pd.concat([table[table["cls"] == c].sample(min(n_per_class, int((table["cls"] == c).sum())), random_state=seed)
                       for c in CLASSES])
    n_rows = (len(picks) + per_row - 1) // per_row
    fig, axes = plt.subplots(n_rows, per_row * 2, figsize=(4 * per_row * 1.6, 3.6 * n_rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    hits = 0
    for k, (_, r) in enumerate(picks.iterrows()):
        rgb = load_upright(r["path"], 512)
        crop, found, (x0, y0, side) = hand_crop(rgb, MARGIN)
        hits += found
        boxed = rgb.copy()
        cv2.rectangle(boxed, (x0, y0), (x0 + side, y0 + side), (0, 255, 0) if found else (255, 0, 0), 6)
        row, col = divmod(k, per_row)
        axes[row, col * 2].imshow(boxed);    axes[row, col * 2].set_title(r["cls"], fontsize=8)
        axes[row, col * 2 + 1].imshow(crop); axes[row, col * 2 + 1].set_title("crop", fontsize=8)
    plt.tight_layout(); plt.show()
    print(f"hand found in {hits} of {len(picks)}")

preview_crops(df)

In [ ]:
WORK_PATH = f"{OUT}/{WORK_FILE}"
X_work = None
if os.path.exists(WORK_PATH):
    z = np.load(WORK_PATH, allow_pickle=True)
    if list(z["rels"]) == list(df["rel"]):
        X_work, found_all = z["X"], z["found"]
        print("loaded", WORK_PATH)
if X_work is None:
    X_work = np.zeros((len(df), WORK_SIZE, WORK_SIZE, 3), np.uint8)
    found_all = np.zeros(len(df), bool)
    t0 = time.time()
    for i, p in enumerate(df["path"]):
        X_work[i], found_all[i] = make_work_copy(load_upright(p, 768))
        if (i + 1) % 200 == 0:
            print(f"{i + 1}/{len(df)} ({time.time() - t0:.0f} s)")
    np.savez(WORK_PATH, X=X_work, found=found_all, rels=df["rel"].values)
df["found"] = found_all
print(f"hand found in {df['found'].mean():.1%} of the photos")
print(df.groupby("cls")["found"].mean().round(3).to_dict())
for fp in (WORK_PATH, SPLIT_FILE):
    if not os.path.exists(f"{PROJECT}/{os.path.basename(fp)}"):
        shutil.copy(fp, PROJECT)

## Augmentation (train only) and the final resize

In [ ]:
def identity_theta(n, device):
    t = torch.zeros(n, 2, 3, device=device)
    t[:, 0, 0] = 1
    t[:, 1, 1] = 1
    return t

def resample(x, theta, out):
    grid = F.affine_grid(theta, (x.size(0), x.size(1), out, out), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)

def augment(x, out):
    n, dev = x.size(0), x.device

    def r(lo, hi):
        return torch.empty(n, device=dev).uniform_(lo, hi)

    ang = r(-math.pi, math.pi)
    s = r(0.65, 1.10)
    f = 1 - 2 * (torch.rand(n, device=dev) < 0.5).float()
    c, sn = torch.cos(ang) * s, torch.sin(ang) * s
    z = torch.zeros_like(c)
    theta = torch.stack([torch.stack([c * f, -sn, z], 1), torch.stack([sn * f, c, z], 1)], 1)
    x = resample(x, theta, out)

    m = x.mean((1, 2, 3), keepdim=True)
    x = (x - m) * r(0.7, 1.3).view(n, 1, 1, 1) + m
    g = x.mean(1, keepdim=True)
    x = (x - g) * r(0.6, 1.4).view(n, 1, 1, 1) + g
    x = (x * torch.empty(n, 3, 1, 1, device=dev).uniform_(0.9, 1.1)).clamp(0, 1)

    low = torch.randint(40, out + 1, (1,)).item()
    pick = (torch.rand(n, device=dev) < 0.5).view(n, 1, 1, 1)
    x_low = F.interpolate(F.interpolate(x, size=(low, low), mode="bilinear", antialias=True, align_corners=False),
                          size=(out, out), mode="bilinear", align_corners=False)
    x = torch.where(pick, x_low, x)

    sigma = torch.rand(n, device=dev) * 1.5 * (torch.rand(n, device=dev) < 0.5).float()
    k = torch.arange(-3, 4, device=dev).float().view(1, 7)
    w1 = torch.exp(-k ** 2 / (2 * sigma.clamp(min=1e-3).view(n, 1) ** 2))
    w1 = (w1 / w1.sum(1, keepdim=True)).repeat_interleave(3, 0)
    xb = F.pad(x, (3, 3, 3, 3), mode="reflect")
    xb = F.conv2d(xb.reshape(1, n * 3, out + 6, out + 6), w1.view(n * 3, 1, 1, 7), groups=n * 3)
    x = F.conv2d(xb, w1.view(n * 3, 1, 7, 1), groups=n * 3).view(n, 3, out, out)

    noise = (torch.rand(n, device=dev) * 0.04 * (torch.rand(n, device=dev) < 0.5).float()).view(n, 1, 1, 1)
    x = (x + torch.randn_like(x) * noise).clamp(0, 1)

    H = W = out
    area = r(0.02, 0.10) * H * W
    asp = r(0.5, 2.0)
    h = torch.sqrt(area * asp).clamp(max=H - 1)
    w = torch.sqrt(area / asp).clamp(max=W - 1)
    y0 = torch.rand(n, device=dev) * (H - h)
    x0 = torch.rand(n, device=dev) * (W - w)
    on = (torch.rand(n, device=dev) < 0.4).view(n, 1, 1)
    ys = torch.arange(H, device=dev).view(1, H, 1)
    xs = torch.arange(W, device=dev).view(1, 1, W)
    box = ((ys >= y0.view(n, 1, 1)) & (ys < (y0 + h).view(n, 1, 1)) &
           (xs >= x0.view(n, 1, 1)) & (xs < (x0 + w).view(n, 1, 1)) & on)
    col = torch.rand(n, 3, 1, 1, device=dev).expand(-1, -1, H, W)
    return torch.where(box.unsqueeze(1), col, x)

In [ ]:
X_all = torch.from_numpy(X_work).permute(0, 3, 1, 2).contiguous()
y_all = torch.tensor(df["cls"].map({c: i for i, c in enumerate(CLASSES)}).values, dtype=torch.long)
split_idx = {s: np.where(df["split"].values == s)[0] for s in ("train", "val", "test")}
print({k: len(v) for k, v in split_idx.items()}, CLASSES)

def show_augmented(n_aug=7, seed=3):
    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(len(CLASSES), n_aug + 1, figsize=(2.2 * (n_aug + 1), 2.5 * len(CLASSES)), squeeze=False)
    for r_, c in enumerate(CLASSES):
        i = rng.choice(np.where((df["cls"].values == c) & (df["split"].values == "train"))[0])
        x = X_all[i:i + 1].float() / 255
        base = resample(x, identity_theta(1, x.device), INPUT_SIZE)
        aug = augment(x.repeat(n_aug, 1, 1, 1), INPUT_SIZE)
        for j, img in enumerate(torch.cat([base, aug])):
            axes[r_][j].imshow(img.permute(1, 2, 0).numpy()); axes[r_][j].axis("off")
            axes[r_][j].set_title(f"{c}: plain" if j == 0 else f"aug {j}", fontsize=8)
    plt.tight_layout(); plt.show()

show_augmented()

## Models

In [ ]:
WIDTHS = [32, 64, 128, 128, 256]

def conv_block(c_in, c_out, k):
    return nn.Sequential(nn.Conv2d(c_in, c_out, k, padding=k // 2), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2))

class Branch(nn.Module):
    def __init__(self, k, blocks):
        super().__init__()
        ch = [3] + WIDTHS[:blocks]
        self.net = nn.Sequential(*[conv_block(ch[i], ch[i + 1], k) for i in range(blocks)],
                                 nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.out = ch[-1]

    def forward(self, x):
        return self.net(x)

class Net(nn.Module):
    def __init__(self, branches, n_classes):
        super().__init__()
        self.branches = nn.ModuleList(branches)
        self.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(sum(b.out for b in branches), 64), nn.ReLU(),
                                  nn.Linear(64, n_classes))

    def forward(self, x):
        return self.head(torch.cat([b(x) for b in self.branches], dim=1))

def build_model(model_type, blocks=3, n_classes=3, input_size=128):
    if model_type == "linear":
        return nn.Sequential(nn.Flatten(), nn.Linear(3 * input_size * input_size, n_classes))
    if model_type == "cnn":
        return Net([Branch(3, blocks)], n_classes)
    if model_type == "twobranch":
        return Net([Branch(3, blocks), Branch(7, blocks)], n_classes)
    raise ValueError(model_type)

for t, b in [("linear", 0), ("cnn", 4), ("cnn", 5), ("twobranch", 5)]:
    print(f"{t:10s} blocks={b}: {sum(p.numel() for p in build_model(t, b).parameters()):,} parameters")

In [ ]:
def save_h5(path, state_dict, meta):
    with h5py.File(path, "w") as f:
        g = f.create_group("weights")
        for name, tensor in state_dict.items():
            g.create_dataset(name, data=tensor.detach().cpu().numpy())
        f.attrs["meta"] = json.dumps(meta)

def load_h5(path):
    with h5py.File(path, "r") as f:
        meta = json.loads(f.attrs["meta"])
        state = {k: torch.as_tensor(np.asarray(f["weights"][k][()])) for k in f["weights"]}
    model = build_model(meta["model"], meta["blocks"], len(meta["classes"]), meta["input_size"])
    model.load_state_dict(state)
    return model.eval(), meta

## Training

In [ ]:
def channel_stats(X_uint8):
    s = s2 = torch.zeros(3, device=X_uint8.device)
    n = 0
    for i in range(0, len(X_uint8), 128):
        x = resample(X_uint8[i:i + 128].float() / 255, identity_theta(len(X_uint8[i:i + 128]), X_uint8.device), INPUT_SIZE)
        s = s + x.sum((0, 2, 3))
        s2 = s2 + (x ** 2).sum((0, 2, 3))
        n += x.shape[0] * x.shape[2] * x.shape[3]
    mean = s / n
    return mean, (s2 / n - mean ** 2).sqrt()

def make_preps(mean, std):
    m, sd = mean.view(1, 3, 1, 1).to(DEVICE), std.view(1, 3, 1, 1).to(DEVICE)

    def prep_eval(xb):
        x = xb.to(DEVICE).float() / 255
        return (resample(x, identity_theta(len(x), DEVICE), INPUT_SIZE) - m) / sd

    def prep_train(xb):
        return (augment(xb.to(DEVICE).float() / 255, INPUT_SIZE) - m) / sd

    return prep_train, prep_eval

@torch.no_grad()
def evaluate(model, X, y, prep, crit):
    model.eval()
    loss, preds = 0.0, []
    for i in range(0, len(X), 128):
        out = model(prep(X[i:i + 128]))
        loss += crit(out, y[i:i + 128]).item() * len(out)
        preds.append(out.argmax(1))
    p, t = torch.cat(preds).cpu().numpy(), y.cpu().numpy()
    return loss / len(t), float((p == t).mean()), float(f1_score(t, p, average="macro")), p, t

def train_model(run_name, model_type, blocks=3, augmentation=True, tr_i=None, va_i=None,
                group="crop-augmentation", save=True):
    torch.manual_seed(SEED)
    tr_i = split_idx["train"] if tr_i is None else tr_i
    va_i = split_idx["val"] if va_i is None else va_i
    te_i = split_idx["test"]
    Xtr, ytr = X_all[tr_i].to(DEVICE), y_all[tr_i].to(DEVICE)
    Xva, yva = X_all[va_i].to(DEVICE), y_all[va_i].to(DEVICE)
    Xte, yte = X_all[te_i].to(DEVICE), y_all[te_i].to(DEVICE)

    mean, std = channel_stats(Xtr)
    prep_train, prep_eval = make_preps(mean, std)
    if not augmentation:
        prep_train = prep_eval

    counts = np.bincount(ytr.cpu().numpy(), minlength=len(CLASSES))
    crit = nn.CrossEntropyLoss(weight=torch.tensor(len(ytr) / (len(CLASSES) * counts), dtype=torch.float32, device=DEVICE))
    model = build_model(model_type, blocks, len(CLASSES), INPUT_SIZE).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)

    meta = {"model": model_type, "blocks": blocks, "classes": CLASSES, "input_size": INPUT_SIZE, "work_size": WORK_SIZE,
            "margin": MARGIN, "mean": mean.cpu().tolist(), "std": std.cpu().tolist(), "color_order": "RGB",
            "augmentation": augmentation,
            "pipeline": "RGB image -> hand crop (skin colour, margin) -> 256x256 -> resize to 128x128 -> (x/255 - mean) / std"}
    wandb.init(project=WANDB_PROJECT, group=group, name=run_name, mode=WANDB_MODE, reinit=True,
               config={**meta, "params": n_params, "n_train": len(tr_i), "n_val": len(va_i), "n_test": len(te_i),
                       "max_epochs": MAX_EPOCHS, "batch_size": BATCH_SIZE, "lr": LR})
    print(f"\n=== {run_name}: {model_type} blocks={blocks} augmentation={augmentation} | "
          f"{n_params:,} params | train {len(tr_i)} val {len(va_i)} test {len(te_i)}")

    best_f1, best_acc, best_state, bad, epochs_run = -1.0, 0.0, None, 0, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(len(Xtr), device=DEVICE)
        t_loss, t_hit = 0.0, 0
        for i in range(0, len(perm), BATCH_SIZE):
            b = perm[i:i + BATCH_SIZE]
            if len(b) < 2:
                continue
            out = model(prep_train(Xtr[b]))
            loss = crit(out, ytr[b])
            opt.zero_grad(); loss.backward(); opt.step()
            t_loss += loss.item() * len(b)
            t_hit += (out.argmax(1) == ytr[b]).sum().item()
        t_loss, t_acc = t_loss / len(perm), t_hit / len(perm)

        v_loss, v_acc, v_f1, _, _ = evaluate(model, Xva, yva, prep_eval, crit)
        sched.step(v_loss)
        wandb.log({"epoch": epoch, "train/loss": t_loss, "train/acc": t_acc, "val/loss": v_loss, "val/acc": v_acc,
                   "val/macro_f1": v_f1, "lr": opt.param_groups[0]["lr"]})
        if epoch == 1 or epoch % 5 == 0:
            print(f"  epoch {epoch:02d} | train {t_loss:.3f} / {t_acc:.3f} | val {v_loss:.3f} / {v_acc:.3f}  F1 {v_f1:.3f}")
        epochs_run = epoch
        if v_f1 > best_f1:
            best_f1, best_acc, bad = v_f1, v_acc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"  early stop at epoch {epoch} (best val F1 {best_f1:.3f})")
                break

    model.load_state_dict(best_state)
    t_loss, t_acc, t_f1, p, t = evaluate(model, Xte, yte, prep_eval, crit)
    print(f"  TEST: accuracy {t_acc:.4f} | macro-F1 {t_f1:.4f}")
    print(classification_report(t, p, labels=list(range(len(CLASSES))), target_names=CLASSES, digits=3, zero_division=0))
    wandb.log({"test/loss": t_loss, "test/acc": t_acc, "test/macro_f1": t_f1,
               "test/confusion_matrix": wandb.plot.confusion_matrix(y_true=t.tolist(), preds=p.tolist(), class_names=CLASSES)})
    wandb.finish()

    res = {"run": run_name, "model": model_type, "blocks": blocks, "augmentation": augmentation,
           "params": n_params, "epochs": epochs_run, "val_acc": round(best_acc, 4), "val_f1": round(best_f1, 4),
           "test_acc": round(t_acc, 4), "test_f1": round(t_f1, 4)}
    if save:
        pth, h5 = f"{OUT}/checkpoints/{run_name}.pth", f"{OUT}/checkpoints/{run_name}.h5"
        torch.save({"state_dict": best_state, **meta}, pth)
        save_h5(h5, best_state, meta)
        os.makedirs(f"{PROJECT}/checkpoints_aug", exist_ok=True)
        shutil.copy(pth, f"{PROJECT}/checkpoints_aug/"); shutil.copy(h5, f"{PROJECT}/checkpoints_aug/")
        res["h5"] = h5
    return res

In [ ]:
experiments = [
    ("crop_linear",     "linear",    0, False),
    ("crop_cnn4",       "cnn",       4, False),
    ("crop_cnn5",       "cnn",       5, False),
    ("aug_cnn4",        "cnn",       4, True),
    ("aug_cnn5",        "cnn",       5, True),
    ("aug_twobranch5",  "twobranch", 5, True),
]
results = []
for name, mtype, blocks, aug in experiments:
    if os.path.exists(f"{PROJECT}/checkpoints_aug/{name}.h5"):
        print("already trained, skipping:", name)
        continue
    results.append(train_model(name, mtype, blocks, aug))
display(pd.DataFrame(results).drop(columns=["h5"]))

## Accuracy and metrics of saved models

In [ ]:
def evaluate_saved(h5_path, part="test", show_plot=True):
    model, meta = load_h5(h5_path)
    model = model.to(DEVICE).eval()
    mean = torch.tensor(meta["mean"], device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor(meta["std"], device=DEVICE).view(1, 3, 1, 1)
    idx = split_idx[part]
    preds, probs = [], []
    with torch.no_grad():
        for i in range(0, len(idx), 128):
            x = X_all[idx[i:i + 128]].to(DEVICE).float() / 255
            x = (resample(x, identity_theta(len(x), DEVICE), meta["input_size"]) - mean) / std
            p = F.softmax(model(x), dim=1)
            probs.append(p.cpu()); preds.append(p.argmax(1).cpu())
    pred, prob, true = torch.cat(preds).numpy(), torch.cat(probs).numpy(), y_all[idx].numpy()
    labels = list(range(len(CLASSES)))

    print(f"=== {os.path.basename(h5_path)} on the {part} images ({len(idx)}) ===")
    print(f"accuracy {accuracy_score(true, pred):.4f} | balanced accuracy {balanced_accuracy_score(true, pred):.4f}")
    pr, rc, f1, sup = precision_recall_fscore_support(true, pred, labels=labels, zero_division=0)
    table = pd.DataFrame({"precision": pr, "recall": rc, "f1": f1, "images": sup}, index=CLASSES).round(3)
    table.loc["macro average"] = [pr.mean().round(3), rc.mean().round(3), f1.mean().round(3), sup.sum()]
    display(table)
    cm = confusion_matrix(true, pred, labels=labels)
    display(pd.DataFrame(cm, index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES]))
    conf = prob.max(1)
    print(f"average confidence: right {conf[pred == true].mean():.2f} | wrong "
          f"{conf[pred != true].mean() if (pred != true).any() else float('nan'):.2f}")

    if show_plot:
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        ax[0].imshow(cm, cmap="Blues"); ax[0].set_title("Confusion matrix (counts)")
        ax[1].imshow(cm / cm.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1); ax[1].set_title("Share of each true class")
        for a, norm in zip(ax, (False, True)):
            a.set_xticks(labels); a.set_xticklabels(CLASSES); a.set_yticks(labels); a.set_yticklabels(CLASSES)
            a.set_xlabel("predicted"); a.set_ylabel("true")
            for i_ in labels:
                for j_ in labels:
                    v = cm[i_, j_] / cm[i_].sum() if norm else cm[i_, j_]
                    a.text(j_, i_, f"{v:.2f}" if norm else int(v), ha="center", va="center",
                           color="white" if (cm[i_, j_] / cm[i_].sum()) > 0.5 else "black")
        plt.tight_layout(); plt.show()
        wrong = np.where(pred != true)[0][:12]
        if len(wrong):
            fig, axs = plt.subplots(1, len(wrong), figsize=(2.2 * len(wrong), 2.8), squeeze=False)
            for a, w in zip(axs[0], wrong):
                a.imshow(X_all[idx[w]].permute(1, 2, 0).numpy()); a.axis("off")
                a.set_title(f"true {CLASSES[true[w]]}\npred {CLASSES[pred[w]]} {prob[w].max():.2f}", fontsize=8)
            plt.suptitle("Some wrong predictions"); plt.show()
    return table

for r in results:
    evaluate_saved(r["h5"], "test")

## Optional: 5-fold cross-validation (train + validation images; the test images stay untouched)

In [ ]:
pool = np.concatenate([split_idx["train"], split_idx["val"]])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = []
for k, (a, b) in enumerate(skf.split(pool, y_all[pool].numpy())):
    cv_results.append(train_model(f"cv5_aug_cnn5_fold{k + 1}", "cnn", 5, True,
                                  tr_i=pool[a], va_i=pool[b], group="cross-validation", save=False))
cv = pd.DataFrame(cv_results)
display(cv[["run", "epochs", "val_acc", "val_f1", "test_acc", "test_f1"]])
print(f"validation accuracy {cv['val_acc'].mean():.4f} +- {cv['val_acc'].std():.4f}")
print(f"test accuracy       {cv['test_acc'].mean():.4f} +- {cv['test_acc'].std():.4f}")

## Download the models

In [ ]:
from google.colab import files
for name in ["crop_cnn4", "aug_cnn4", "aug_cnn5"]:
    for ext in ("h5", "pth"):
        files.download(f"{OUT}/checkpoints/{name}.{ext}")